# Class Activity: SQL, Byond Basic 4/29
 - Linking tables with Foreign Keys.
-  Combining data with INNER, LEFT, and OUTER JOINs.
- Summarizing data with GROUP BY, SUM, and AVG.
- Updating records and Altering Table Schema.


In [47]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:') # :memory: means whatever you do it's available during this session only, not affecting the database
conn.execute("PRAGMA foreign_keys = ON;") # lets read between tables -- define column as reference to another table
cursor = conn.cursor()



## Practice Join

In [48]:
#executescript is for multiple lines of command (separated by semicolon) here it's TABLE dept and TABLE employees
cursor.executescript('''
CREATE TABLE depts (dept_id INTEGER PRIMARY KEY, dept_name TEXT);
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, 
    name TEXT, 
    dept_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES depts(dept_id)
);
''') # dept_id is the bridge between the tables, so it is the FOREIGN KEy that references the other table 
# this means that if you want to update your employee table, you can't give the new employee a dept_id that doesn't exist in the depts table

cursor.execute("INSERT INTO depts VALUES (1, 'Engineering'), (2, 'Marketing'), (3, 'Finance')") # insertig values directly

cursor.execute("INSERT INTO employees VALUES (101, 'Alice', 1), (102, 'Bob', 2), (103, 'Charlie', NULL)")

In [49]:
print("INNER JOIN: Matches Only\n")

sql ="SELECT e.name, d.dept_name FROM employees e JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn)) # using alias e and d for employees and depts tables
# won't print finance -- no employees, nor Charlie, noo department

print('#'*50)
print("\n LEFT JOIN: All Employees\n") 
sql = "SELECT e.name, d.dept_name FROM employees e LEFT JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

print('#'*50)
print("\n RIGHT JOIN: All Departments\n")
sql = "SELECT e.name, d.dept_name FROM employees e RIGHT JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

print('#'*50)
print("\n FULL OUTER JOIN: The Whole Picture\n")
sql = "SELECT e.name, d.dept_name FROM employees e FULL OUTER JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

print('#'*50)
print("\n FULL OUTER JOIN: The Whole Picture\n")
sql = "SELECT * FROM employees e FULL OUTER JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))
# use star to get the whole table

INNER JOIN: Matches Only

    name    dept_name
0  Alice  Engineering
1    Bob    Marketing
##################################################

 LEFT JOIN: All Employees

      name    dept_name
0    Alice  Engineering
1      Bob    Marketing
2  Charlie         None
##################################################

 RIGHT JOIN: All Departments

    name    dept_name
0  Alice  Engineering
1    Bob    Marketing
2   None      Finance
##################################################

 FULL OUTER JOIN: The Whole Picture

      name    dept_name
0    Alice  Engineering
1      Bob    Marketing
2  Charlie         None
3     None      Finance
##################################################

 FULL OUTER JOIN: The Whole Picture

   emp_id     name  dept_id  dept_id    dept_name
0   101.0    Alice      1.0      1.0  Engineering
1   102.0      Bob      2.0      2.0    Marketing
2   103.0  Charlie      NaN      NaN         None
3     NaN     None      NaN      3.0      Finance


In [50]:
try:
    cursor.execute("INSERT INTO employees VALUES (11, 'John', 5)")
except sqlite3.IntegrityError as e:
    print(f"BLOCKED: {e} (Could be attempting to add an employee with no valid department)")
# foreign key was dept_id that didn't exist in the depts table

try:
    cursor.execute("INSERT INTO employees VALUES (11, 'John', 3)")
except sqlite3.IntegrityError as e:
    print(f"BLOCKED: {e} (Could be attempting to add an employee with no valid department)")
# using 3 for dept_id works fine and doesn't trigger the error message as dept_id = 3 exists

BLOCKED: FOREIGN KEY constraint failed (Could be attempting to add an employee with no valid department)


### Aggregate Functions (Summarizing Data)
SUM, COUNT, AVG, MIN, MAX


In [51]:
# Create Categories and Products Tables
cursor.execute("CREATE TABLE IF NOT EXISTS categories (id INTEGER PRIMARY KEY, cat_name TEXT)") # categories, not felines
cursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY, 
        name TEXT, 
        price REAL, 
        cat_id INTEGER,
        FOREIGN KEY (cat_id) REFERENCES categories(id)
    )""")

# Insert DATA
cursor.execute("INSERT OR REPLACE INTO categories VALUES (1, 'Electronics'), (2, 'Furniture')")
cursor.executemany("INSERT INTO products (name, price, cat_id) VALUES (?,?,?)", [
    ('Laptop', 1200.00, 1),
    ('Phone', 800.00, 1),
    ('Desk', 250.00, 2),
    ('Chair', 150.00, 2),
    ('Monitor', 300.00, 1)
])

In [52]:
print("BASIC GROUP BY (Count products per category)")

query1 = """
SELECT c.cat_name, COUNT(p.id) as total_products
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
"""
print(pd.read_sql_query(query1, conn))


print("\nFINANCIAL SUMMARY (Average & Total Price)")

query2 = """
SELECT c.cat_name, 
       SUM(p.price) as inventory_value,
       ROUND(AVG(p.price), 2) as average_price
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
"""
#  those commands happen from bottom to top, priority wise
print(pd.read_sql_query(query2, conn))


BASIC GROUP BY (Count products per category)
      cat_name  total_products
0  Electronics               3
1    Furniture               2

FINANCIAL SUMMARY (Average & Total Price)
      cat_name  inventory_value  average_price
0  Electronics           2300.0         766.67
1    Furniture            400.0         200.00


 ### Advanced Analytics Using the HAVING Filter
 'WHERE' filters rows before grouping. 'HAVING' filters the results AFTER grouping.

In [53]:
query3 = """
SELECT c.cat_name, SUM(p.price) as total_value
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
HAVING total_value > 500
""" # slightly off from my priority understanding, but also not? 
print(pd.read_sql_query(query3, conn))


      cat_name  total_value
0  Electronics       2300.0


### Which category has the single most expensive item?

In [54]:
query4 = """
SELECT c.cat_name, MAX(p.price) as max_value
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
ORDER BY max_value DESC
LIMIT 2
""" # limit 2 is returning two records, (it would return one for each category without that). If limit was 1, it would only return the maximum value
print(pd.read_sql_query(query4, conn))

      cat_name  max_value
0  Electronics     1200.0
1    Furniture      250.0


### Extra Activity
Use
"ALTER TABLE table_name ADD COLUMN column_name data_type"
1. To add "quantity" and "discount" column to product tables.
2. update this values for each product
3. find the product with quantity less than 5 
4. Find the categories with maximum discount
5. What is the total inventory and total inventroy value?

In [55]:
print("Add quantity and discount columns to products \n")
cursor.execute("ALTER TABLE products ADD COLUMN quantity INTEGER")
cursor.execute("ALTER TABLE products ADD COLUMN discount REAL")



Add quantity and discount columns to products 



In [56]:
print("update values for each product\n")
cursor.execute("UPDATE products SET quantity = 3, discount = .25 WHERE name = 'Laptop'")
cursor.execute("UPDATE products SET quantity = 6, discount = .15 WHERE name = 'Phone'")
cursor.execute("UPDATE products SET quantity = 4, discount = .50 WHERE name = 'Desk'")
cursor.execute("UPDATE products SET quantity = 7, discount = .20 WHERE name = 'Chair'")
cursor.execute("UPDATE products SET quantity = 1, discount = NULL WHERE name = 'Monitor'")


update values for each product



In [57]:
df = pd.read_sql_query("SELECT * FROM products", conn)
print(df)

   id     name   price  cat_id  quantity  discount
0   1   Laptop  1200.0       1         3      0.25
1   2    Phone   800.0       1         6      0.15
2   3     Desk   250.0       2         4      0.50
3   4    Chair   150.0       2         7      0.20
4   5  Monitor   300.0       1         1       NaN


In [58]:
print("Find products with quantity less than five \n")
df = pd.read_sql_query("SELECT * FROM products WHERE quantity < 5", conn)
print(df)

Find products with quantity less than five 

   id     name   price  cat_id  quantity  discount
0   1   Laptop  1200.0       1         3      0.25
1   3     Desk   250.0       2         4      0.50
2   5  Monitor   300.0       1         1       NaN


In [59]:
print("Find Category with the most Discount \n")

discountquery = """
SELECT c.cat_name, MAX(p.discount) as max_discount
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
ORDER BY max_discount DESC
"""

print(pd.read_sql_query(discountquery, conn))

Find Category with the most Discount 

      cat_name  max_discount
0    Furniture          0.50
1  Electronics          0.25


In [65]:
print("Total inventory, total value \n")
total_query = """
SELECT 
    SUM(p.quantity) as total_inventory, 
    SUM(p.price * p.quantity) as total_ value
FROM products p
"""

print(pd.read_sql_query(total_query, conn))

Total inventory, total value 



DatabaseError: Execution failed on sql '
SELECT 
    SUM(p.quantity) as total_inventory, 
    SUM(p.price * p.quantity) as total_ value
FROM products p
': near "value": syntax error